# 🐝 Embodied Hornet — Spiking SLAM + Neuromechanical Flight Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lhooz/embodied_hornet/blob/main/notebooks/demo_colab.ipynb)

This notebook demonstrates the **Embodied Hornet** framework: a biologically-grounded autonomous micro-aerial vehicle that combines:

| Layer | What it does |
|:---|:---|
| **Neuromechanical hover specialist** | Pre-trained IDA-PBC ICNN that keeps the hornet aloft using passivity-preserving control |
| **Spiking SLAM system** | Event-camera + 3-beam ToF sensor fusion for online mapping |
| **SHAC navigation policy** | Short-Horizon Actor-Critic trained with DNAG (Dynamic Neural Attention Gate) that learns to navigate obstacle rooms |

---
**Pipeline:**
1. ⚙️ **Setup** — Clone repo (with submodules), install deps
2. 🦟 **Phase 1** *(optional)* — Train the hover specialist with hornetRL
3. 🗺️ **Phase 2** — Train SHAC navigation on the 1 m × 1 m obstacle arena
4. 🎬 **Replay** — Visualize the trained policy with SLAM sensor overlay

> 💡 **Tip:** Switch Runtime → GPU for ~10× faster training. The hover specialist (`hover_params.pkl`) is included in the repo so you can skip Phase 1.

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive + Clone Repository
# ============================================================
from google.colab import drive
import os, subprocess

print("→ Mounting Google Drive for checkpoint persistence...")
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/EmbodiedHornet_Dev'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"→ Working directory: {PROJECT_ROOT}")

REPO_NAME = 'embodied_hornet'
REPO_URL  = 'https://github.com/lhooz/embodied_hornet.git'

if not os.path.exists(REPO_NAME):
    print("→ Cloning embodied_hornet (with submodules)...")
    # --recurse-submodules pulls hornetRL, fly_surrogate, neuro-symbolic-slam
    !git clone --recurse-submodules {REPO_URL}
else:
    print("→ Repo already exists. Pulling latest...")
    os.chdir(REPO_NAME)
    !git pull
    !git submodule update --init --recursive
    os.chdir('..')

REPO_PATH = os.path.join(PROJECT_ROOT, REPO_NAME)
os.chdir(REPO_PATH)
print(f"→ Repo ready at: {REPO_PATH}")

In [ ]:
# ============================================================
# CELL 2 — Install Dependencies
# ============================================================
# Strategy:
#  • pip install -e .  ← installs Python deps (jax, dm-haiku, optax, etc.)
#  • sys.path entries  ← wires up the three submodule source trees directly
#
# Why not pip install -e hornetRL / neuro-symbolic-slam?
#  • hornetRL's pinned pyproject.toml has a duplicate [tool.setuptools.packages.find]
#    key that pip rejects with a TOML parse error
#  • neuro-symbolic-slam has no pyproject.toml at its root (sources live in src/)
#  Both are cleanly handled by sys.path — the same approach used by
#  embodied_hornet/__init__.py at runtime.
# ============================================================
import os, sys

if 'REPO_PATH' not in dir():
    REPO_PATH = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
os.chdir(REPO_PATH)

# 1. Install embodied_hornet — pulls jax, dm-haiku, optax, numpy, matplotlib
print("→ Installing Python dependencies via embodied_hornet package...")
!pip install -e . -q

# 2. Wire submodule source trees directly into sys.path
#    (embodied_hornet/__init__.py does this at runtime too, but we
#     also need it in the notebook kernel for direct imports in Cell 7)
print("→ Adding submodule paths to sys.path...")
_submodule_paths = [
    os.path.join(REPO_PATH, 'hornetRL'),                     # fly_system, fluid_surrogate, neural_cpg
    os.path.join(REPO_PATH, 'fly_surrogate'),                # aerodynamic surrogate
    os.path.join(REPO_PATH, 'neuro-symbolic-slam', 'src'),   # snn_slam_system, sparse_forest
]
for _p in _submodule_paths:
    if _p not in sys.path:
        sys.path.insert(0, _p)
    print(f"   ✓ {os.path.relpath(_p, REPO_PATH)}")

# 3. Quick import smoke-test
print("\n→ Verifying imports...")
try:
    import jax
    import haiku as hk
    import optax
    from fly_system import FlappingFlySystem     # noqa
    from sparse_forest import generate_obstacles  # noqa
    from snn_slam_system import SNNSLAMSystem     # noqa
    print("   ✅ All imports OK")
except ImportError as e:
    print(f"   ⚠️  Import error: {e}")
    print("   Check that --recurse-submodules completed successfully in Cell 1.")

# 4. GPU check
devices = jax.devices()
print(f"\n→ JAX devices: {devices}")
HAS_GPU = any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in devices)
print(f"→ GPU available: {'✅ Yes' if HAS_GPU else '❌ No — see note below'}")
if not HAS_GPU:
    print("   Training will be ~40 s/step on CPU.")
    print("   → Runtime → Change runtime type → T4 GPU for 10× speedup.")

In [ ]:
# ============================================================
# CELL 3 — Configuration
# ============================================================
import os
if 'REPO_PATH' not in dir():
    REPO_PATH = '/content/drive/MyDrive/EmbodiedHornet_Dev/embodied_hornet'
os.chdir(REPO_PATH)

# ── User-configurable ──────────────────────────────────────
USE_GPU           = True      # Set False to force CPU
PHASE1_STEPS      = 2000     # Hover specialist training steps (skip if hover_params.pkl exists)
PHASE2_STEPS      = 100000   # Navigation SHAC training steps (set lower for a quick demo)
QUICK_DEMO_STEPS  = 500      # Steps for a quick sanity-check run
# ───────────────────────────────────────────────────────────

CKPT_DIR       = os.path.join(REPO_PATH, 'checkpoints_shac')
HOVER_PKL      = os.path.join(REPO_PATH, 'embodied_hornet', 'hover_params.pkl')
GPU_FLAG       = '--gpu' if USE_GPU else ''

os.makedirs(CKPT_DIR, exist_ok=True)

print("=" * 50)
print("        Embodied Hornet — Demo Config")
print("=" * 50)
print(f"  Repo path     : {REPO_PATH}")
print(f"  Checkpoint dir: {CKPT_DIR}")
print(f"  GPU mode      : {USE_GPU}")
print(f"  Hover PKL     : {'✅ found' if os.path.exists(HOVER_PKL) else '❌ missing (Phase 1 needed)'}")

import glob, re
existing = sorted(glob.glob(os.path.join(CKPT_DIR, 'shac_params_*.pkl')),
                  key=lambda f: int(re.search(r'_(\d+)', f).group(1)))
if existing:
    latest = existing[-1]
    step   = int(re.search(r'_(\d+)', latest).group(1))
    print(f"  Latest ckpt   : {os.path.basename(latest)} (step {step}) — will resume")
else:
    print(f"  Latest ckpt   : none — will train from scratch")

In [ ]:
# ============================================================
# CELL 4 — Phase 1: Hover Specialist (skip if already present)
# ============================================================
# The hover specialist is a pre-trained IDA-PBC ICNN that keeps
# the hornet aloft. It is included in the repo (hover_params.pkl),
# so this cell is SKIPPED automatically if the file exists.
#
# If for any reason it is missing, this cell trains it from
# scratch using the hornetRL training script (~30 min on GPU T4).
# ============================================================
import os, shutil

if os.path.exists(HOVER_PKL):
    print(f"✅ hover_params.pkl already present — skipping Phase 1.")
    print(f"   Path: {HOVER_PKL}")
else:
    print("⚠️  hover_params.pkl not found. Running Phase 1 hover training...")
    print(f"   This will train for {PHASE1_STEPS} steps (~30 min on T4 GPU).\n")

    # hornetRL's training produces hover_params.pkl in the checkpoints directory
    HOVER_CKPT_DIR = os.path.join(REPO_PATH, 'hornetRL', 'checkpoints_shac')
    os.makedirs(HOVER_CKPT_DIR, exist_ok=True)

    !python -m hornetRL.train {GPU_FLAG} \
        --dir "{HOVER_CKPT_DIR}" \
        --steps {PHASE1_STEPS}

    # Copy the best hover checkpoint to where embodied_hornet expects it
    hover_src = os.path.join(HOVER_CKPT_DIR, 'hover_params.pkl')
    if os.path.exists(hover_src):
        shutil.copy(hover_src, HOVER_PKL)
        print(f"\n✅ hover_params.pkl copied to: {HOVER_PKL}")
    else:
        print("\n❌ Phase 1 did not produce hover_params.pkl. Check logs above.")

In [ ]:
# ============================================================
# CELL 5 — Phase 2: Navigation Training (SHAC + SLAM + DNAG)
# ============================================================
# Trains the SHAC navigation policy on a 1 m × 1 m obstacle
# arena with 15 random obstacles.
#
# Key components:
#  • DNAG gate blends hover specialist (frozen) with nav policy
#  • 3-beam ToF + event-camera SLAM provides spatial awareness
#  • Obstacle collision: gradient penalty + episode termination
#  • Checkpoint auto-resume if shac_params_*.pkl exists
#
# Training is launched as an inline subprocess so you see live
# output. Interrupt (⏹) to pause — re-run this cell to resume.
# ============================================================
import os
os.chdir(REPO_PATH)

print("→ Starting Phase 2: Navigation Training")
print(f"   Max steps : {PHASE2_STEPS}")
print(f"   Checkpoint: {CKPT_DIR}")
print(f"   GPU mode  : {USE_GPU}")
print()

# Runs training. Interrupt (⏹) at any time — it will resume from
# the latest checkpoint next time this cell is executed.
!python -m embodied_hornet.train {GPU_FLAG} 2>&1 | tee /tmp/train_out.txt

In [ ]:
# ============================================================
# CELL 5b — Quick Demo Run (≈ 5 min)
# ============================================================
# Runs only QUICK_DEMO_STEPS steps, then generates a
# visualization GIF automatically. Good for a quick sanity check.
# Comment out CELL 5 above and run this cell instead.
# ============================================================
# Uncomment the block below to use:

# import os
# os.chdir(REPO_PATH)
#
# # Temporarily patch total steps for the quick demo
# import embodied_hornet.train as _t
# _t.Config.TOTAL_UPDATES = QUICK_DEMO_STEPS
# _t.Config.VIS_INTERVAL  = QUICK_DEMO_STEPS  # generate GIF at end
#
# !python -m embodied_hornet.train {GPU_FLAG} 2>&1 | tee /tmp/train_out.txt

print("(Quick demo cell — uncomment to use)")

In [ ]:
# ============================================================
# CELL 6 — Replay: Display Latest Epoch GIF
# ============================================================
# Shows the most recent epoch_*.gif from the checkpoint directory.
# The GIF has two panels:
#   LEFT  — SLAM navigation room: trajectory, 3 ToF beams,
#            camera FOV cone, obstacles (red = collision)
#   RIGHT — Wing mechanics close-up with nodal aerodynamic forces
# ============================================================
import os, glob, re, shutil
from IPython.display import Image, display

os.chdir(REPO_PATH)

if 'CKPT_DIR' not in dir():
    CKPT_DIR = os.path.join(REPO_PATH, 'checkpoints_shac')

gifs = sorted(glob.glob(os.path.join(CKPT_DIR, 'epoch_*.gif')),
              key=lambda f: int(re.search(r'epoch_(\d+)', f).group(1)))

if not gifs:
    # Also check project root
    gifs = sorted(glob.glob(os.path.join(REPO_PATH, 'epoch_*.gif')),
                  key=lambda f: int(re.search(r'epoch_(\d+)', f).group(1)))

if not gifs:
    print("⚠️  No epoch GIFs found yet. Run the training cell first.")
else:
    latest_gif = gifs[-1]
    step = int(re.search(r'epoch_(\d+)', latest_gif).group(1))
    print(f"→ Displaying: {os.path.basename(latest_gif)} (training step {step})")

    # Copy to /tmp for display (Drive paths can be slow to serve)
    tmp_gif = f'/tmp/replay_step_{step}.gif'
    shutil.copy(latest_gif, tmp_gif)

    print(f"\n{'='*60}")
    print(f"  SLAM Navigation Room (left) + Wing Mechanics (right)")
    print(f"  Step {step} | ToF beams: blue=left/right, yellow=center")
    print(f"{'='*60}\n")
    display(Image(filename=tmp_gif, width=900))

    if len(gifs) > 1:
        print(f"\n→ {len(gifs)} epoch GIFs available: ",
              [int(re.search(r'epoch_(\d+)', g).group(1)) for g in gifs])
        print("  Edit 'gifs[-1]' → 'gifs[0]' to compare with the earliest epoch.")

In [ ]:
# ============================================================
# CELL 7 — Training Progress Summary
# ============================================================
# Parses the tee'd training log (/tmp/train_out.txt) to plot
# loss, reward, and force-match error over training steps.
# Cell 5 already redirects output there with '| tee /tmp/train_out.txt'
# ============================================================
import re, os, glob
import numpy as np
import matplotlib.pyplot as plt

LOG_FILE = '/tmp/train_out.txt'

if not os.path.exists(LOG_FILE):
    print(f"Log file not found at {LOG_FILE}.")
    print("Cell 5 tees output there automatically when run.")
else:
    steps, losses, rewards, frcs = [], [], [], []
    pat = re.compile(
        r'Step (\d+).*Loss: ([\d.e+\-]+) .*Rew: ([\d.\-]+).*Frc:([\d.]+)'
    )
    with open(LOG_FILE) as f:
        for line in f:
            m = pat.search(line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))
                rewards.append(float(m.group(3)))
                frcs.append(float(m.group(4)))

    if not steps:
        print("No Step log lines found yet — training may still be starting.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.patch.set_facecolor('#0d0d0d')
        for ax in axes:
            ax.set_facecolor('#111111')
            ax.tick_params(colors='#aaaaaa')
            for sp in ax.spines.values(): sp.set_edgecolor('#444444')

        axes[0].plot(steps, losses, color='#ff6666', lw=1.5)
        axes[0].set_title('Total Loss', color='white')
        axes[0].set_xlabel('Step', color='#aaaaaa')

        axes[1].plot(steps, rewards, color='#66ff99', lw=1.5)
        axes[1].set_title('Mean Reward', color='white')
        axes[1].set_xlabel('Step', color='#aaaaaa')

        axes[2].plot(steps, frcs, color='#ffaa44', lw=1.5)
        axes[2].set_title('Force Match Error (Frc)', color='white')
        axes[2].set_xlabel('Step', color='#aaaaaa')
        axes[2].set_yscale('log')

        plt.suptitle(f'Embodied Hornet — Training Progress ({len(steps)} steps)',
                     color='white', fontsize=13)
        plt.tight_layout()
        plt.show()

        print(f"\nSteps logged : {len(steps)}")
        print(f"Loss range   : {min(losses):.1f} – {max(losses):.1f}")
        print(f"Best reward  : {max(rewards):.1f}")
        print(f"Frc trend    : {frcs[0]:.4f} → {frcs[-1]:.4f}  "
              f"({'improved ✅' if frcs[-1] < frcs[0] else 'not yet ⏳'})")